# 📈 05 - Benchmark Complet

Benchmark approfondi de tous les modèles avec analyse statistique.

**M2 MoSEF - Université Paris 1 Panthéon-Sorbonne**

In [ ]:
import sys
sys.path.insert(0, '..')

import io
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from app.services.captcha_generator import CaptchaGenerator
from app.services.solver_service import SolverService

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
generator = CaptchaGenerator()
solver = SolverService()

print(f"🤖 Modèles disponibles: {solver.available_models}")

## 1. Configuration du Benchmark

In [ ]:
# Paramètres du benchmark
BENCHMARK_CONFIG = {
    'n_samples': 50,           # Nombre d'échantillons par configuration
    'noise_levels': [0.1, 0.3, 0.5],  # Niveaux de bruit à tester
    'lengths': [4, 5, 6],      # Longueurs à tester
    'models': ['trocr'],       # Modèles à tester (ajouter 'crnn', 'florence' si disponibles)
}

print("📋 Configuration du benchmark:")
for key, value in BENCHMARK_CONFIG.items():
    print(f"   {key}: {value}")

## 2. Exécution du Benchmark

In [ ]:
def run_benchmark(config):
    """Exécute le benchmark complet."""
    results = []
    
    total_tests = (
        config['n_samples'] * 
        len(config['noise_levels']) * 
        len(config['lengths']) * 
        len(config['models'])
    )
    
    print(f"🚀 Lancement de {total_tests} tests...")
    
    with tqdm(total=total_tests) as pbar:
        for noise in config['noise_levels']:
            for length in config['lengths']:
                for i in range(config['n_samples']):
                    # Générer le CAPTCHA
                    image, true_text = generator.generate(
                        length=length,
                        noise_level=noise,
                    )
                    
                    buffer = io.BytesIO()
                    image.save(buffer, format="PNG")
                    image_bytes = buffer.getvalue()
                    
                    # Tester chaque modèle
                    for model in config['models']:
                        try:
                            result = solver.solve(image_bytes, model=model)
                            
                            results.append({
                                'model': model,
                                'noise': noise,
                                'length': length,
                                'true_text': true_text,
                                'prediction': result.text,
                                'correct': result.text.lower() == true_text.lower(),
                                'confidence': result.confidence,
                                'time_ms': result.processing_time_ms,
                            })
                        except Exception as e:
                            results.append({
                                'model': model,
                                'noise': noise,
                                'length': length,
                                'true_text': true_text,
                                'prediction': '',
                                'correct': False,
                                'confidence': None,
                                'time_ms': None,
                                'error': str(e),
                            })
                        
                        pbar.update(1)
    
    return pd.DataFrame(results)

# Exécuter
df = run_benchmark(BENCHMARK_CONFIG)
print(f"\n✅ Benchmark terminé: {len(df)} résultats")

## 3. Analyse des Résultats

In [ ]:
# Accuracy globale par modèle
accuracy_global = df.groupby('model')['correct'].mean() * 100

print("📊 Accuracy globale par modèle:")
for model, acc in accuracy_global.items():
    print(f"   {model.upper()}: {acc:.1f}%")

In [ ]:
# Accuracy par niveau de bruit
accuracy_noise = df.pivot_table(
    values='correct',
    index='model',
    columns='noise',
    aggfunc='mean'
) * 100

plt.figure(figsize=(10, 5))
accuracy_noise.T.plot(kind='bar', width=0.8)
plt.title('Accuracy par niveau de bruit')
plt.xlabel('Niveau de bruit')
plt.ylabel('Accuracy (%)')
plt.legend(title='Modèle')
plt.ylim(0, 100)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Accuracy par longueur
accuracy_length = df.pivot_table(
    values='correct',
    index='model',
    columns='length',
    aggfunc='mean'
) * 100

plt.figure(figsize=(10, 5))
accuracy_length.T.plot(kind='bar', width=0.8)
plt.title('Accuracy par longueur de CAPTCHA')
plt.xlabel('Longueur')
plt.ylabel('Accuracy (%)')
plt.legend(title='Modèle')
plt.ylim(0, 100)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des temps d'exécution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.boxplot(data=df, x='model', y='time_ms')
plt.title('Distribution du temps d\'exécution')
plt.ylabel('Temps (ms)')

plt.subplot(1, 2, 2)
sns.violinplot(data=df, x='model', y='time_ms')
plt.title('Distribution (violin plot)')
plt.ylabel('Temps (ms)')

plt.tight_layout()
plt.show()

## 4. Heatmap Accuracy

In [ ]:
# Heatmap bruit x longueur pour chaque modèle
for model in df['model'].unique():
    model_df = df[df['model'] == model]
    
    heatmap_data = model_df.pivot_table(
        values='correct',
        index='noise',
        columns='length',
        aggfunc='mean'
    ) * 100
    
    plt.figure(figsize=(8, 5))
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt='.1f',
        cmap='RdYlGn',
        vmin=0,
        vmax=100,
        cbar_kws={'label': 'Accuracy (%)'}
    )
    plt.title(f'Accuracy {model.upper()} (Bruit x Longueur)')
    plt.xlabel('Longueur')
    plt.ylabel('Niveau de bruit')
    plt.tight_layout()
    plt.show()

## 5. Rapport Final

In [ ]:
# Statistiques détaillées
summary = df.groupby('model').agg({
    'correct': ['sum', 'count', 'mean', 'std'],
    'time_ms': ['mean', 'std', 'min', 'max'],
    'confidence': ['mean', 'std']
}).round(3)

print("📊 RAPPORT DE BENCHMARK")
print("=" * 60)
print(f"\nÉchantillons totaux: {len(df)}")
print(f"Niveaux de bruit: {BENCHMARK_CONFIG['noise_levels']}")
print(f"Longueurs: {BENCHMARK_CONFIG['lengths']}")
print("\n")

for model in df['model'].unique():
    model_df = df[df['model'] == model]
    acc = model_df['correct'].mean() * 100
    time_mean = model_df['time_ms'].mean()
    time_std = model_df['time_ms'].std()
    
    print(f"🤖 {model.upper()}")
    print(f"   Accuracy: {acc:.1f}%")
    print(f"   Temps moyen: {time_mean:.0f} ± {time_std:.0f} ms")
    print()

In [ ]:
# Sauvegarder les résultats
df.to_csv('benchmark_results.csv', index=False)
print("💾 Résultats sauvegardés: benchmark_results.csv")

---
**Fin du notebook 05**